# ĐỒ ÁN THỰC HÀNH #2

## Giai đoạn 4: Building Model + Evaluation + Bonus

## Môn học: Phân tích dữ liệu ứng dụng - CSC12110

Các thành viên và bảng đánh giá:

| STT | MSSV     | Họ tên           | Công việc đã thực hiện trong giai đoạn 4 | Tỉ lệ công việc (trên tổng số 100%) | Đánh giá (thang 100%) | Ghi chú |
| --- | ----     | ---------------- | ---------------------- | --------------- | --------------------- | ------- |
| 1   | 22127012 | Nguyễn Hà Anh    | Build model Logistic Regression | 25% | 100% |  |
| 2   | 22127205 | Bùi Lê Khôi      | Build model SVM, phân tích và nhận xét các mô hình | 25% | 100% | Leader |
| 3   | 22127260 | Bùi Công Mậu     | Build model Random Forest | 25% | 100% |  |
| 4   | 22127400 | Thái Hữu Thọ     | Build model Decision Tree, setup tiền xử lí cho mô hình | 25% | 100% |  |


---

Link sản phẩm:

- Drive: [DA_05_S1_2526](https://drive.google.com/drive/u/0/folders/1TPJKcKhVyfVeW6RG4pF22pNRB-k1RA9r)
  - Giai đoạn 1 - Data Cleaning & Data Preprocessing: [DA_05_GD1_Preprocessing](https://colab.research.google.com/drive/12gd8Hqaj5c117MPsPFZiHgXDFbH3MjDn)
  - Giai đoạn 2 - EDA: [DA_05_GD2_EDA](https://colab.research.google.com/drive/1nnAVOIJJaglbqq1lOBo9OiKlEMXFFqnO)
  - Giai đoạn 3 - Hypothesis Testing: [DA_05_GD3_Hypothesis_Testing](https://colab.research.google.com/drive/1gVtnBd4bN6eDq8xDgmxVu3IEwPYaTBsa)
  - Giai đoạn 4 - DA_05_GD4_Model_Evaluation + Bonus Tasks: [DA_05_GD4_Model_Evaluation+Bonus](https://colab.research.google.com/drive/1DE0dN8WIMKHM_72EjdzM_iXx8nAXOCLM)

- Video Demo: [Final_DA_05](https://drive.google.com/file/d/1cvEYJuuxS5Zg5s358HRbRD8DQqvk3WBg/view)

---

### Cập nhật 05/01/2025:

- Thêm giải thích về các tham số build thêm lấy từ đâu? (phần 4.1 - Xử lí outlier và nhiễu)

---
---


## Mô tả đồ án

- Mục tiêu: **Dự đoán khả năng một người có mắc bệnh tim hay không**.

    - Biến mục tiêu: ```target``` (1 = bệnh, 0 = không bệnh).

- Dataset: [Heart Disease UCI Dataset](https://www.kaggle.com/datasets/redwankarimsony/heart-disease-data)

---

### Import các libs và dependencies

In [ ]:
import re
import os
import math

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import openpyxl as pxl
import seaborn as sns
import statsmodels.api as sm
import datetime as dt

from datetime import timedelta
from scipy import stats
from math import sqrt
from tqdm.notebook import tqdm


from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report, pairwise_distances, silhouette_score, precision_score, recall_score, f1_score, roc_auc_score,  accuracy_score, make_scorer
from sklearn.metrics import RocCurveDisplay, roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.feature_selection import RFE
from sklearn.model_selection import train_test_split, cross_val_score,  cross_validate
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from imblearn.over_sampling import SMOTE
from collections import Counter
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_selection import RFECV

from google.colab import drive
import warnings
warnings.filterwarnings("ignore")

# Ensure plots appear in the notebook
%matplotlib inline

# Print last updated timestamp
import time
print(f"Last updated: {time.asctime()}")

Last updated: Wed Dec 31 16:58:27 2025


In [ ]:
# authorization
drive.mount('/content/drive')

# Check if the directory exists and list files
folder_path = '/content/drive/MyDrive/DA_05_S1_2526'

if os.path.exists(folder_path):
    print("Folder found. Files inside:")
    print(os.listdir(folder_path))
else:
    print(f"Folder not found at: {folder_path}")

Mounted at /content/drive
Folder found. Files inside:
['PTDLUD_HK1_2526_DATH_nopass.pdf', 'Ôn thi cuối kì.gdoc', 'heart_disease_uci.csv', 'DA_05_GD4_Model_Evaluation+Bonus.ipynb', 'heart_disease_uci_preprocessed.csv', 'DA_05_GD1_Preprocessing.ipynb', 'DA_05_GD2_EDA.ipynb', 'heart_disease_uci_with_target.csv', 'DA_05_GD3_Hypothesis_Testing.ipynb']


---
---

# Giai đoạn 4: Building Model + Evaluation + Bonus

## 4.0. Đọc dữ liệu

In [ ]:
# Read preprocessed csv file
df = pd.read_csv('/content/drive/MyDrive/DA_05_S1_2526/heart_disease_uci_with_target.csv')
# Check the data
df

,id,age,sex,location,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num,target
0,1,63,1,Cleveland,1,145.0,233.0,1,2,150.0,0,2.3,3,0,7,0,0
1,2,67,1,Cleveland,4,160.0,286.0,0,2,108.0,1,1.5,2,3,3,2,1
2,3,67,1,Cleveland,4,120.0,229.0,0,2,129.0,1,2.6,2,2,6,1,1
3,4,37,1,Cleveland,3,130.0,250.0,0,0,187.0,0,3.5,3,0,3,0,0
4,5,41,0,Cleveland,2,130.0,204.0,0,2,172.0,0,1.4,1,0,3,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
915,916,54,0,VA Long Beach,4,127.0,333.0,1,1,154.0,0,0.0,2,0,3,1,1
916,917,62,1,VA Long Beach,1,132.0,139.0,0,1,138.0,0,0.5,2,0,3,0,0
917,918,55,1,VA Long Beach,4,122.0,223.0,1,1,100.0,0,0.0,2,0,7,2,1
918,919,58,1,VA Long Beach,4,132.0,385.0,1,2,138.0,0,0.5,2,0,3,0,0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 920 entries, 0 to 919
Data columns (total 17 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   id        920 non-null    int64  
 1   age       920 non-null    int64  
 2   sex       920 non-null    int64  
 3   location  920 non-null    object 
 4   cp        920 non-null    int64  
 5   trestbps  920 non-null    float64
 6   chol      920 non-null    float64
 7   fbs       920 non-null    int64  
 8   restecg   920 non-null    int64  
 9   thalach   920 non-null    float64
 10  exang     920 non-null    int64  
 11  oldpeak   920 non-null    float64
 12  slope     920 non-null    int64  
 13  ca        920 non-null    int64  
 14  thal      920 non-null    int64  
 15  num       920 non-null    int64  
 16  target    920 non-null    int64  
dtypes: float64(4), int64(12), object(1)
memory usage: 122.3+ KB


### Mô tả dữ liệu cho giai đoạn 4 (```heart_disease_uci_with_target.csv```)

- Dữ liệu có 920 dòng và 17 cột.

- Các **đặc trưng đầu vào** trong dataset trên gồm:
    - ```id```: Mã số duy nhất cho mỗi bệnh nhân (```int```).
    - ```age```: Tuổi của bệnh nhân (```int```).
    - ```sex```: Giới tính (Male - Nam = ```1``` /Female - Nữ = ```0```).
    - ```location```: Nơi nghiên cứu. Giá trị nằm trong tập {Cleveland, Hungary, Switzerland, VA Long Beach}
    - ```cp```: Loại đau thắt ngực. Giá trị gồm:
        - ```typical angina```: đau thắt ngực điển hình = ```1```
        - ```atypical angina```: đau thắt ngực không điển hình = ```2```
        - ```non-anginal```: không đau thắt ngực = ```3```
        - ```asymptomatic```: không triệu chứng = ```4```
    - ```trestbps```: Huyết áp lúc nghỉ (tính bằng mmHg khi nhập viện) (```float```)
    - ```chol```: Nồng độ cholesterol huyết thanh tính bằng mg/dl (```float```).
    - ```fbs```: Trả về ```1``` nếu đường huyết lúc đói > 120 mg/dl, ngược lại trả về ```0```.
    - ```restecg```: Kết quả điện tâm đồ lúc nghỉ: Giá trị gồm:
        - ```normal```: bình thường = ```0```
        - ```stt abnormality```: bất thường ST-T = ```1```
        - ```lv hypertrophy```: phì đại thất trái = ```2```
    - ```thalach```: nhịp tim tối đa đạt được (```float```)
    - ```exang```: Trả về ```1``` nếu có đau thắt ngực do gắng sức, ngược lại trả về ```0```.
    - ```oldpeak```: ST chênh xuống do gắng sức so với lúc nghỉ (```float```)
    - ```slope```: độ dốc của đỉnh đoạn ST khi gắng sức. Giá trị gồm:
        - ```upsloping```: Đi lên. Bình thường nếu giá trị dương nhỏ, bất thường nếu dương lớn = ```1```
        - ```flat```: Nằm ngang. Thường sẽ đc chẩn đoán là thiếu máu cục bộ cơ tim = ```3```
        - ```downsloping```: Đi xuống. Là dấu hiệu cảnh báo tình trạng thiếu máu cơ tim nghiêm trọng hoặc tổn thương cơ tim ```3```
    - ```ca```: số mạch máu chính được tô màu bằng huỳnh quang. Giá trị nằm trong tập {0, 1, 2, 3}.
    - ```thal```: Giá trị gồm:
        - ```normal```: bình thường = ```3```
        - ```fixed defect```: khuyết cố định = ```6```
        - ```reversable defect```: khuyết có thể hồi phục = ```7```
    - ```num```: thuộc tính dự đoán (```int```), trong đó:
        - ```0```: không có bệnh tim.
        - ```1```: có bệnh tim mức độ nhẹ.
        - ```2```: có bệnh tim mức độ trung bình.
        - ```3```: có bệnh tim mức độ nặng.
        - ```4```: có bệnh tim mức độ rất nặng.
    - ```target```: biến mục tiêu (```int```), trong đó:
        - ```0```: không có bệnh tim.
        - ```1```: có bệnh tim.


    Về mặt lí thuyết ```trestbps``` và ```thalach``` nhận giá trị nguyên không âm. Tuy nhiên trong tính toán có thể dẫn đến các số lẻ và cần làm tròn, do đó ta để kiểu dữ liệu ```float``` cho thuận tiện.

- **Phân loại đặc trưng**:
    - Các đặc trưng định lượng (Quantitative):
        - Định lượng liên tục (Continuous): ```trestbps```, ```chol```, ```thalach```, ```oldpeak```, ```age```.
        - Định lượng rời rạc (Discrete): ```ca```.
    - Các đặc trưng định tính (Qualitative):
        - Định tính danh nghĩa (Nominal): ```id```, ```sex```, ```dataset```, ```cp```, ```restecg```, ```slope```, ```thal```, ```target```.
        - Định tính thứ tự (Ordinal): ```fbs```, ```exang```, ```num```.

---
---

## 4.1. Xây dựng các mô hình


Trong phần này ta xây dựng các mô hình học máy để dự đoán bệnh tim dựa trên các đặc trung lâm sàng.

Bài toán có dạng **binary classification**, với ```target``` = 0 là việc không mắc bệnh tim, và ```target``` = 1 là có mắc bệnh tim.

Mục tiêu là xây dựng được 4 mô hình:
- Mô hình Baseline: Logistic Regression.
- Mô hình so sánh:
    - Decision Tree.
    - Random Forest.
    - Support Vector Machine (SVM).
sau đó so sánh hiệu quả của chúng và chọn ra mô hình tốt nhất.


### Xử lí outlier và nhiễu
Để giảm ảnh hưởng và sai lệch trong quá trình train các mô hình, đặc biệt là các mô hình nhạy cảm với nhiễu như LR và SVM, ta cần xử lí để loại bỏ các giá trị ngoại lại.

Ta dùng phương pháp Isolation Forest, một phương pháp phát hiện outlier không giám sát (unsupervised). IF hoạt động tốt trên cả dữ liệu đa chiều, phù hợp với tập dữ liệu có gần 20 thuộc tính này.

Chọn ```contamination``` = 5% để không loại bỏ quá nhiều thông tin, đảm bảo giữ lại được nhiều nhất có thể các thông tin y khoa quan trọng.

Các thuộc tính gốc của dữ liệu có thể không đủ để phản ánh dữ liệu, do vậy ta tạo thêm một số thuộc tính có ý nghĩa sinh học để tìm được các mối quan hệ tiềm ẩn:

- ```hr_diff```(Heart Rate Difference - HRD):
    - Được tính bằng khoảng chênh lệch giữa nhịp tim tối đa lúc gắng sức trên lý thuyết (HRMax) và giá trị thực tế đo được trên bệnh nhân (```thalach```): HRR = HRMax - ```thalach```.
    - HRMax có thể được ước lượng bằng công thức Maximum Heart Rate (MHR): 220 - ```age```.
    - Chỉ số này càng nhỏ, tim càng gần "tới ngưỡng", cho thấy cơ thể thích nghi yếu hơn; ngược lại càng lớn thì tim càng có nhiều khả năng đáp ứng, là dấu hiệu cho thấy sức khỏe tốt hơn.
    - Tham khảo ở [Revascularization among patients with acute myocardial infarction complicated by cardiogenic shock and impact of American College of Cardiology/American Heart Association guidelines](https://pubmed.ncbi.nlm.nih.gov/15541246/). Ngoài ra, có thể tính bằng công thức Haskell & Fox. Tool này giúp tính toán nhanh giá trị trên: [Max Heart Rate Calculator](https://www.omnicalculator.com/sports/max-heart-rate)
- ```risk_index```:
    - Ta coi các dấu hiệu nhận biết trực quan sau là nguy cơ mắc bệnh hiện hữu:
        - ```cp``` = 4: Đau thắt ngực không triệu chứng.
        - ```exang``` = 1: Đau thắt ngực khi gắng sức.
    - Với mỗi dấu hiệu, ta cộng 1 điểm vào chỉ số risk_index. Như vậy ta có:
        - ```risk_index``` = 0: Không có yếu tố nguy cơ rõ ràng.
        - ```risk_index``` = 1: Có một yếu tố.
        - ```risk_index``` = 2: Có cả hai yếu tố, là nguy cơ cao.
- ```pulse_pressure```:
    - Ước lượng huyết áp tâm trương. Dataset này chỉ có huyết áp tâm thu (cột ```trestbps```), do đó ta ước lượng tương đối bằng cách trừ giá trị của ```trestbps``` cho 45 (thường chênh lệch giữa 2 trị số này trong khoảng 40 - 50 mmHg).
    - ```pulse_pressure`` đánh giá khả năng đàn hồi của thành mạch và áp lực máu lên thành động mạch khi tim giãn ra. Chỉ số này cao phản ánh các nguy cơ liên quan đến tăng huyết áp, xở vữa động mạch, ...; trong khi chỉ số này quá thấp thường là dấu hiệu của bệnh huyết áp thấp, suy tim, rối loạn chuyển hóa, ... .
    - Tham khảo cách ước lượng ở [Chênh lệch huyết áp tâm thu và tâm trương: Phân biệt chi tiết](https://tamanhhospital.vn/chenh-lech-huyet-ap-tam-thu-va-tam-truong/)


In [ ]:
# Loại bỏ 5% dữ liệu gây nhiễu + new features
iso = IsolationForest(contamination=0.05, random_state=42)

# Các cột cần loại bỏ nhiễu
outliers = iso.fit_predict(df[['age', 'trestbps', 'chol', 'thalach', 'oldpeak']])
df = df[outliers == 1]

# Thêm các thuộc tính tiềm ẩn
df['hr_diff'] = (220 - df['age']) - df['thalach']
df['risk_index'] = ((df['cp'] == 4).astype(int) + (df['exang'] == 1).astype(int))
df['pulse_pressure'] = df['trestbps'] - 45


---

### Chuẩn bị dữ liệu và xử lí mất cân bằng dữ liệu

Sau khi có đủ các thuộc tính cần thiết, ta chuẩn bị dữ liệu chuẩn để đưa vào các thuật toán học máy, gồm:

- Tách dữ liệu thành ```X``` (tập đặc trưng) và ```y``` (biến mục tiêu - ```target```)

- Mã hoá biến phân loại ```location```.

- Chia tập dữ liệu:
    - Tỉ lệ 8:2 (Train chiếm 80%, Test chiếm 20%)
    - Sử dụng ```stratify=y``` để giữ nguyên tỷ lệ lớp.

- Chuẩn hoá dữ liệu bằng StandardScaler, đặc biệt cần thiết cho Logistic Regression và SVM.

- Xử lí mất cân bằng dữ liệu bằng phương pháp SMOTE (Synthetic Minority Over-sampling Technique), trong đó:

    - Tạo các điểm dữ liệu tổng hợp cho lớp thiểu số.
    - Giúp mô hình không thiên lệch về lớp đa số.
    - Chỉ áp dụng SMOTE trên tập train để tránh data leakage.

Bằng cách này, ta xử lí được vấn đề mất cân bằng dữ liệu (class imbalance) mà trong đó số bệnh nhân không bệnh chiếm ưu thế hơn nhóm có bệnh.



In [ ]:
# Chuẩn bị X, y, Encode Location, split, SMOTE, scaling
X = df.drop(columns=['id', 'num', 'target'])
y = df['target']
X['location'] = LabelEncoder().fit_transform(X['location'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled = scaler.transform(X_test)


---

### Huấn luyện các mô hình và tối ưu hyperparameter

Ta huấn luyện 4 mô hình phân lớp như đã nói ở trên.

Các bước thực hiện bao gồm:
#### 1. Khai báo:
Ta cấu hình cho từng mô hình trong 1 dictionary ```model_configs```. Mỗi mô hình sẽ có tên, đối tượng mô hình ```model```, các siêu tham số cần tối ưu ```param```.

Các parameter cần tối ưu của mỗi mô hình:
- Logistic Regression: là baseline, có khả năng diễn giải cao.
    - ```C```: ```[0.001, ..., 1000]```, là nghịch đảo của hệ số regularization. ```C``` càng nhỏ thì regularization càng lớn và mô hình càng đơn giản nhưng dễ bị underffitting; ngược lại ```C``` càng lớn thì mô hình càng học được kĩ nhưng dễ bị overffitting.
    - ```penalty```: ```['l1', 'l2']```
        - ```l1```: Giúp feature selection tự nhiên (nhiều hệ số về 0)
        - ```l2```: Làm mượt trọng số.
    - ```solver```: ```liblinear```, do nó hỗ trợ cả ```l1``` và ```l2``` và phù hợp với tập dữ liệu nhỏ (dataset gốc chỉ có 920 dòng).
    Ta tối ưu LR để làm baseline, có khả năng diễn giải cao.
- Decision Tree:
    - ```criterion```: Do độ thuần khiết của node ```gini``` hoặc ```entropy```.
    - ```max_depth```: Thử các giá trị độ sâu để xem hiện tượng overfitting.
    - ```min_samples_split```và ```min_samples_leaf```: Tránh việc tạo node từ quá ít mẫu (giảm nhiễu).
    DT là mô hình có thể bắt được mối quan hệ phi tuyến (non-linear), dễ diễn giải, nhưng cần kiểm soát tham số tốt để tránh overffitting.
- Random Forest:
    - ```n_estimators```: Số lượng "cây trong rừng". Nhều cây hơn thì tốn thời gian train hơn, đổi lại được mô hình ổn định hơn.
    - ```max_depth```, ```min_samples_split```: Kiểm soát overfitting ở từng cây.
    - ```max_features```: Chỉ định số đặc trưng xét tại mỗi split, giúp tăng tính đa dạng giữa các cây.
    Random Forest có khả năng diễn giải kém hơn các mô hình khác, nhưng đổi lại nó có hiệu suất cao và ít bị ảnh hưởng bởi nhiễu.
- SVM:
    - ```C```: Điều chỉnh mức độ cho phép sai số.
    - ```gamma```: Quyết định phạm vi ảnh hưởng của một điểm dữ liệu đơn lẻ.
        - ```gamma``` thấp: "Tầm với" của các điểm dữ liệu xa hơn. Một điểm dữ liệu sẽ có ảnh hưởng rộng đến việc phân loại các điểm ở xa nó. Điều này tạo ra ranh giới quyết định (decision boundary) mượt mà, ít phức tạp hơn.
        - ```gamma``` cao: "Tầm với" của các điểm dữ liệu ngắn lại. Mô hình chỉ tập trung vào các điểm ở gần ranh giới. Điều này tạo ra ranh giới quyết định rất uốn lượn và bám sát các điểm dữ liệu.
    - ```kernel```: ```linear``` khi dữ liệu đã phân tách tuyến tính; ```rbf``` cho quan hệ phi tuyến và số chiều lớn; ```sigmoid``` để làm một hàm kích hoạt phi tuyến khác cho SVM.
    SVM sẽ hiệu quả khi mô hình có số chiều đặc trưng cao, biên phân tách phức tạp.

#### 2. Thực hiện GridSearchCV với cross-validation

Ta setup với bộ tham số:
```py
grid = GridSearchCV(
    config['model'],
    config['params'], # thử tất cả các tổ hợp tham số trong param
    cv=5, # 5-fold cross-validation, chia tỉ lệ train:test = 4:1
    scoring='accuracy', # để so sánh tổng thể
    n_jobs=-1 # kích toàn bộ CPU để chạy mô hình
)
```

#### 3. Lưu lại mô hình tốt nhất cho từng thuật toán

Sau khi GridSearch hoàn tất, lưu lại từng mô hình tốt nhất của mỗi loại vào dictionary ```best_estimators```. Các model này sẽ được dùng để chạy trên tập test và cho ra kết quả độ đo mô hình để đánh giá.

In [ ]:
# Model configs + GridSearchCV
model_configs = {
    "Logistic Regression": {
        "model": LogisticRegression(max_iter=5000, random_state=42),
        "params": {
            'C': [0.001, 0.01, 0.1, 1, 10, 100, 1000],
            'penalty': ['l1', 'l2'],
            'solver': ['liblinear']
        }
    },
    "Decision Tree": {
        "model": DecisionTreeClassifier(random_state=42),
        "params": {
            'criterion': ['gini', 'entropy'],
            'max_depth': [3, 5, 8, 12, 15, 20, None],
            'min_samples_split': [2, 5, 10, 20],
            'min_samples_leaf': [1, 2, 4, 8]
        }
    },
    "Random Forest": {
        "model": RandomForestClassifier(random_state=42),
        "params": {
            'n_estimators': [100, 200, 500, 800],
            'max_depth': [5, 10, 15, 20, None],
            'min_samples_split': [2, 5, 10],
            'max_features': ['sqrt', 'log2', None]
        }
    },
    "SVM": {
        "model": SVC(probability=True, random_state=42),
        "params": {
            'C': [0.1, 1, 10, 100, 500],
            'gamma': ['scale', 'auto', 1, 0.1, 0.01, 0.001, 0.0001],
            'kernel': ['rbf', 'linear', 'sigmoid']
        }
    }
}

best_estimators = {}

# Dùng tqdm để theo dõi tiến độ GridSearch cho từng model
for name, config in tqdm(list(model_configs.items()), desc="GridSearch models"):
    grid = GridSearchCV(config['model'], config['params'], cv=5, scoring='accuracy', n_jobs=-1)
    grid.fit(X_train_scaled, y_train_res)
    best_estimators[name] = grid.best_estimator_
    print(f"\nModel: {name}")
    print(f"Best Params: {grid.best_params_}")
    print(f"Best Cross-validation Score: {grid.best_score_:.4f}")
    print("-" * 30)


GridSearch models:   0%|          | 0/4 [00:00<?, ?it/s]


Model: Logistic Regression
Best Params: {'C': 10, 'penalty': 'l2', 'solver': 'liblinear'}
Best Cross-validation Score: 0.8299
------------------------------

Model: Decision Tree
Best Params: {'criterion': 'gini', 'max_depth': 3, 'min_samples_leaf': 8, 'min_samples_split': 2}
Best Cross-validation Score: 0.7867
------------------------------

Model: Random Forest
Best Params: {'max_depth': 15, 'max_features': 'sqrt', 'min_samples_split': 10, 'n_estimators': 800}
Best Cross-validation Score: 0.8273
------------------------------

Model: SVM
Best Params: {'C': 10, 'gamma': 'scale', 'kernel': 'linear'}
Best Cross-validation Score: 0.8325
------------------------------


---
---

## 4.2. Chạy trên dữ liệu test và đo các độ đo kết quả

In [ ]:
# Cell 5: Evaluate models, tìm threshold, build summary (có tqdm để track)
summary_final = []

for name, model in tqdm(list(best_estimators.items()), desc="Evaluate models"):
    # Lấy xác suất dự đoán
    y_probs = model.predict_proba(X_test_scaled)[:, 1]

    # Tìm ngưỡng tốt nhất để đạt Accuracy tối đa trên tập Test
    fpr, tpr, thresholds = roc_curve(y_test, y_probs)

    # sử dụng tqdm cho vòng lặp thresholds (để thấy tiến độ khi thresholds lớn)
    accuracies = []
    for t in tqdm(thresholds, desc=f"Thresholds - {name}", leave=False):
        accuracies.append(accuracy_score(y_test, (y_probs >= t).astype(int)))

    best_t = thresholds[np.argmax(accuracies)]

    # Dự đoán với ngưỡng mới
    y_pred_final = (y_probs >= best_t).astype(int)

    # Tính toán các thông số yêu cầu
    acc = accuracy_score(y_test, y_pred_final)
    prec = precision_score(y_test, y_pred_final)
    rec = recall_score(y_test, y_pred_final)
    f1 = f1_score(y_test, y_pred_final)
    auc = roc_auc_score(y_test, y_probs)

    # Summary
    summary_final.append({
        "Mô hình": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1,
        "AUC": auc,
        "Best Threshold": best_t
    })

    print(f"\n{name} (Threshold: {best_t:.2f})")
    print(f"Accuracy: {acc:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")

df_summary = pd.DataFrame(summary_final).sort_values(by='Accuracy', ascending=False)
print("\nSummary:")
print(df_summary.to_string(index=False))


Evaluate models:   0%|          | 0/4 [00:00<?, ?it/s]

Thresholds - Logistic Regression:   0%|          | 0/48 [00:00<?, ?it/s]


Logistic Regression (Threshold: 0.48)
Accuracy: 0.8457 | F1: 0.8657 | AUC: 0.9233


Thresholds - Decision Tree:   0%|          | 0/9 [00:00<?, ?it/s]


Decision Tree (Threshold: 0.66)
Accuracy: 0.7943 | F1: 0.8105 | AUC: 0.8520


Thresholds - Random Forest:   0%|          | 0/40 [00:00<?, ?it/s]


Random Forest (Threshold: 0.68)
Accuracy: 0.8400 | F1: 0.8372 | AUC: 0.9229


Thresholds - SVM:   0%|          | 0/48 [00:00<?, ?it/s]


SVM (Threshold: 0.54)
Accuracy: 0.8514 | F1: 0.8687 | AUC: 0.9202

Summary:
            Mô hình  Accuracy  Precision   Recall  F1-Score      AUC  Best Threshold
                SVM  0.851429   0.843137 0.895833  0.868687 0.920227        0.542165
Logistic Regression  0.845714   0.828571 0.906250  0.865672 0.923259        0.479329
      Random Forest  0.840000   0.947368 0.750000  0.837209 0.922864        0.684928
      Decision Tree  0.794286   0.819149 0.802083  0.810526 0.851991        0.663366


---
---

## 4.3. Nhận xét
> Nếu không nói gì thêm, trong phần này các mô hình Logistic Regression, Decision Tree, Random Forest và Support Vector Machine được gọi tắt là LR, DT, RF và SVM.

### 4.3.1. Nhận xét theo từng chỉ số:

- Ta thấy về Accuracy, các mô hình SVM, LR, và RF đều hoạt động khác tốt khi có giá trị từ 84% - 85.14%. Điều đó cho thấy dữ liệu đã được tiền xử lí tốt. DT cho hiệu suất hơi kém hơn một chút, 79,43%, có thể giải thích là do DT đơn lẻ dễ bị overfitting hơn các mô hình khác, cũng không ổn định như RF và cũng không có tính tối ưu cao như SVM hay LR.

- Về Precision, RF cho chỉ số cao hơn hẳn (94.74%) so với các mô hình còn lại (dao động từ 81.91% - 84.31%). Điều này dễ hiểu vì RF đánh đổi khả năng diễn giải để đổi lấy hiệu suất cao và chất lượng dự đoán dương tính (positive) cho mô hình.

- Xét chỉ số Recall ta có thể chia làm 2 nhóm:
    - SVM và LR có chỉ số này cao (90.63% và 89.58%), đảm bảo việc phát hiện ra nhiều nhất các trường hợp positive.
    - DT và RF có chỉ số kém hơn hẳn. DT chỉ đạt khoảng 80%, trong khi RF còn tệ hơn, chỉ 75%. Đây là cái đánh đổi mà ta đã nói ở phần Precision.
    
- F1-score của SVM và LR cũng gần như ngang nhau (86.87% và 86.57%), cho thấy sự cân bằng hiệu quả giữa Precision và Recall. Kém hơn chút là RF với 83.72%, khi chỉ số Precision "gánh" cho mô hình này. DT kém hẳn ở cả 2 chỉ số và chỉ đạt F1-score là 81.05%.

- Trong khả năng phân loại - AUC, 3 mô hình SVM, LR và RF đều đạt trong khoảng hơn 92%, cho thấy khả năng phân loại hiệu quả giữa ca "bệnh" và "không bệnh" tốt. Trong khi đó, DT kém hơn với chỉ 85.2%, dễ hiểu vì nó là mô hình hướng tới sự đơn giản, dễ diễn giải nhưng phải đánh đổi hiệu suất.

- Một điểm bất ngờ khác trong bảng kết quả này nằm ở việc trong 4 mô hình, chỉ có 2 trong đó chọn ngưỡng Best Threshold ở mức gần với 0.5. Cụ thể
    - SVM chọn mức Best Threshold là khoảng 0.54, trong khi với LR là xấp xỉ 0.48. Ta nhận thấy 2 mô hình này đều là tuyến tính (dựa theo kết quả thu được ở trên để xét cho SVM), và chúng có xác suát dự đoán khá chuẩn, đường ranh giới quyết định của các mô hình khá rõ ràng.
    - Ngược lại RF có chỉ số này 0.685, và đối với DT là 0.66, bị đẩy lên lệch hẳn về 1 phía. Để đạt được F1-score tối ưu, 2 mô hình này sẵn sàng đòi hỏi xác suất dự đoán lên cao để đảm bảo chốt được một trường hợp nào đó là có bệnh hay không.


### 4.3.2. Nhận xét theo từng mô hình:

- Với SVM, mô hình này hướng tới việc cân bằng giữa Precision và Recall. Ta thấy các chỉ số của nó rất đồng đều, dao động từ 84% - 92%. Trong y học, chỉ số Recall là tối quan trọng, đảm bảo tỉ lệ phát hiện số người bệnh trên tổng thể người bệnh cao nhất có thể, và SVM làm rất tốt khi đạt xấp xỉ 89.6%, ngang ngửa với LR. Precision cũng khá ổn khi đạt 84.3%, hạn chế khá tốt tình trạng "dự đoán có bênh nhưng lại không có bệnh", là một việc quan trọng để tránh lãng phí tài nguyên chữa bệnh. SVM cũng có khả năng phân loại tốt khi đạt 92%.

- Xét đên LR, mô hình này có các chỉ số gần như ngang ngửa với SVM, với Recall cao hơn một tí, Precision nhỏ hơn một chút, các chỉ số còn lại như Accuracy, F1-score, AUC khá đồng đều. Khác biệt đáng kể nhất nằm ở Threshold, khi SVM có giá trị này đạt khoảng 0.54 trong khi LR thì là 0.48. Điều này phản ánh rằng SVM sẽ hơi kén hơn một tí, sự chắc chắn phải đạt khoảng 54% thì mới dám phân loại một ca bệnh là Positive, trong khi LR thì sẽ nới hơn, chỉ cần đạt xác suất 48% là đã có thể kết luận.

- Nhìn chung, 2 mô hình này đều tối ưu khá hiệu quả và có thể sử dụng để làm mô hình dự đoán bệnh tim. Tuy nhiên, xét về tổng thể, ta vẫn kết luận rằng SVM là mô hình chiến thắng. LR về nhì với thành tích rất sát sao.
    
- Trong môi trường thực tế, nếu cần tốc độ rất cao mà vẫn đảm bảo hiệu suất tương đối, người ta cũng hay chọn LR vì dễ triển khai, build/train/test mô hình, tốc độ tính toán tốt, dễ diễn giải và không cần nhiều bước chuẩn hóa dữ liệu phức tạp. SVM thường được ưu tiên hơn khi có phần cứng đảm bảo, có thời gian tinh chỉnh mô hình trước khi triển khai vì độ phức tạp của nó cao hơn.

- Đối với RF, mô hình này đánh đổi lấy Precision rất cao (gần 95%), đảm bảo gần như "ca nào mô hình kết luận có bệnh thì sẽ có bệnh". Tuy nhiên Recall là chỉ số phải trả giá khi có giá trị thấp nhất trong các mô hình, chỉ đạt 75%, tức là nếu lấy trung bình, cứ mỗi 4 bệnh nhân bị bệnh thì RF sẽ bỏ sót mất 1 người. Ta thấy được tính chất quan trọng của RF là thận trọng khi đưa ra quyết định, đã chốt có bệnh thì là có bệnh, nhưng mô hình sẽ chấp nhận việc bỏ sót một số trường hợp. Điều này sẽ có lợi thế khi cần xét nghiệm số lượng rất lớn các mẫu nếu như nó là bài toán của lĩnh vực nào đó, nhưng với y học, cụ thể là mô hình dự đoán bệnh tim, một vấn đề cần chính xác cao, thì sẽ cần cân nhắc rất kỹ việc có nên dùng mô hình này hay không.

- DT là mô hình đơn giản dễ hiểu nhất, nhưng đổi lại khả năng tổng quát của nó bị hạn chế khi so với các mô hình còn lại. Điều này giải thích cho việc các chỉ số của nó hầu hết đều thấp hơn 3 mô hình còn lại. Việc này chỉ ra rằng một cây quyết định thuần đúng/sai sẽ không đủ sức để mô hình hóa hết mối quan hệ phức tạo giữa các biến trong bài toán về y học, về mặt bằng chung có thể diễn giải được, nhưng càng vào chi tiết DT càng tỏ ra lép vế.

### 4.3.3. Kết luận chung:
- SVM là mô hình hiệu quả nhất trong các mô hình đã khảo sát, theo sát là LR. RF có thể hữu ích trong một vài bối cảnh, còn DT là mô hình nhìn chung kém hiệu quả nhất.
- Trên thực tế, người ta có thể dùng SVM và tinh chỉnh thêm để tăng hiểu quả, hoặc kết hợp giữa SVM và LR, dùng những kĩ thuật như Ensemble Voting (Soft Voting) chẳng hạn.
    





---
---

<center>HẾT</center>